# Ablation and sensitivity sweeps

Needs `results/records.csv` (`python -m taintwatch.evaluation`) and `results/sweeps.csv`
(`python -m taintwatch.analysis sweeps`).

Spotlight rows depend on an assumed resistance probability, and the scorer weights are
hand-written, so read the curves as properties of these baselines and not of detectors in general.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from taintwatch.analysis import pareto_front, read_points_csv
from taintwatch.evaluation import benign_breakdown, read_records

ROOT = Path.cwd()
if not (ROOT / 'results').exists():
    ROOT = ROOT.parent
points = read_points_csv(ROOT / 'results' / 'sweeps.csv')
scorer = sorted((p for p in points if p.defense.startswith('scorer@')), key=lambda p: p.param or 0)
spotlight = sorted(
    (p for p in points if p.defense.startswith('spotlight@')), key=lambda p: p.param or 0
)
refs = [p for p in points if p.param is None]
print(len(points), 'points:', len(scorer), 'scorer,', len(spotlight), 'spotlight,',
      len(refs), 'reference')

## Attack success against benign utility (top left is best)

In [ ]:
def xy(items):
    return ([p.attack_success.value * 100 for p in items],
            [p.benign_utility.value * 100 for p in items])


fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(*xy(scorer), marker='o', label='scorer (threshold sweep)')
for p in scorer:
    ax.annotate(f'{p.param:g}', (p.attack_success.value * 100, p.benign_utility.value * 100),
                textcoords='offset points', xytext=(4, 4), fontsize=8)
ax.plot(*xy(spotlight), marker='s', label='spotlight (resistance sweep)')
for p in refs:
    x, y = xy([p])
    ax.scatter(x, y, marker='*', s=140, label=p.defense)
ax.set_xlabel('attack success (%), lower is better')
ax.set_ylabel('benign utility (%), higher is better')
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Spotlight sensitivity

In [ ]:
xs = [p.param for p in spotlight]
ys = [p.attack_success.value * 100 for p in spotlight]
rates = [p.attack_success for p in spotlight]
low = [max(0.0, (r.value - r.interval()[0]) * 100) for r in rates]
high = [max(0.0, (r.interval()[1] - r.value) * 100) for r in rates]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.errorbar(xs, ys, yerr=[low, high], marker='o', capsize=3)
ax.set_xlabel('assumed resistance probability')
ax.set_ylabel('attack success (%)')
fig.tight_layout()
plt.show()

## Pareto front

In [ ]:
for p in pareto_front(points):
    attack, benign = p.attack_success.percent(), p.benign_utility.percent()
    print(f'{p.defense:<20} attack {attack:>7}  benign {benign:>7}')

## Ablation: span-level versus whole-context labels

Both defenses stop the same attacks. The difference is utility: labeling a call with the taint
of the whole context rejects tasks that only forward trusted values.

In [ ]:
records = read_records(ROOT / 'results' / 'records.csv')
span = benign_breakdown(records, 'taintwatch')
coarse = benign_breakdown(records, 'taintwatch-coarse')
print(f"{'family':<24}{'span-level':>12}{'whole-context':>16}")
for family in span:
    print(f'{family:<24}{span[family].percent():>12}{coarse[family].percent():>16}')